# Análisis de resultados Gold

Este notebook consulta las tablas Gold creadas por `src/gold.ipynb`. No crea ni modifica tablas. Las consultas describen asociaciones observadas entre Twitch y Steam; no permiten afirmar causalidad.

In [ ]:
dbutils.widgets.text("catalog", "steam_game_intelligence")
dbutils.widgets.text("gold_schema", "gold")
catalog = dbutils.widgets.get("catalog")
gold_schema = dbutils.widgets.get("gold_schema")

## 1. Cobertura de las tablas Gold

Muestra el volumen y las ventanas temporales disponibles para saber qué datos sustentan las consultas posteriores.

In [ ]:
%sql
SELECT 'game_growth' AS table_name, COUNT(*) AS row_count,
       COUNT(DISTINCT app_id) AS distinct_games,
       MIN(month_start) AS first_month,
       MAX(month_start) AS last_month
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.game_growth')
UNION ALL
SELECT 'twitch_steam_relationship', COUNT(*), COUNT(DISTINCT app_id),
       MIN(month_start), MAX(month_start)
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.twitch_steam_relationship')
UNION ALL
SELECT 'dim_game', COUNT(*), COUNT(DISTINCT app_id), NULL, NULL
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_game');

## 2. Juegos con mayor crecimiento mensual

Resume los juegos y meses con mayor crecimiento consecutivo. Se excluyen valores nulos y se limita la muestra para facilitar la revisión.

In [ ]:
%sql
SELECT app_id, name, month_start, avg_players,
       prev_avg_players, mom_change_pct, months_since_release
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.game_growth')
WHERE mom_change_pct IS NOT NULL
ORDER BY mom_change_pct DESC, app_id, month_start
LIMIT 50;

## 3. Evolución mensual agregada

Calcula el promedio y la mediana de jugadores por mes. Sirve para observar la evolución general sin confundirla con el comportamiento de un solo juego.

In [ ]:
%sql
SELECT month_start,
       COUNT(DISTINCT app_id) AS games_with_measurement,
       AVG(avg_players) AS mean_avg_players,
       percentile_approx(avg_players, 0.5) AS median_avg_players,
       AVG(mom_change_pct) AS mean_mom_change_pct
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.game_growth')
GROUP BY month_start
ORDER BY month_start;

## 4. Asociación entre Twitch y el crecimiento posterior en Steam

Agrupa las observaciones con jugadores disponibles en `t` y `t+1`. La correlación y los promedios son descriptivos: no controlan por lanzamientos, promociones, estacionalidad ni otros factores.

In [ ]:
%sql
SELECT
  COUNT(*) AS observations,
  COUNT(DISTINCT app_id) AS distinct_games,
  AVG(hours_watched) AS mean_hours_watched,
  AVG(growth_pct_t_to_t1) AS mean_steam_growth_t_to_t1,
  percentile_approx(growth_pct_t_to_t1, 0.5) AS median_steam_growth_t_to_t1,
  corr(hours_watched, growth_pct_t_to_t1) AS corr_hours_growth
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.twitch_steam_relationship')
WHERE avg_players_t IS NOT NULL
  AND avg_players_t1 IS NOT NULL
  AND growth_pct_t_to_t1 IS NOT NULL;

## 5. Asociación por nivel de exposición en Twitch

Divide las observaciones en cuartiles de horas vistas y compara el crecimiento posterior de Steam entre grupos. El resultado sigue siendo asociativo, no causal.

In [ ]:
%sql
WITH observations AS (
  SELECT *, ntile(4) OVER (ORDER BY hours_watched, app_id, month_start) AS exposure_quartile
  FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.twitch_steam_relationship')
  WHERE avg_players_t IS NOT NULL
    AND avg_players_t1 IS NOT NULL
    AND growth_pct_t_to_t1 IS NOT NULL
)
SELECT exposure_quartile,
       COUNT(*) AS observations,
       MIN(hours_watched) AS min_hours_watched,
       MAX(hours_watched) AS max_hours_watched,
       AVG(growth_pct_t_to_t1) AS mean_growth_t_to_t1,
       percentile_approx(growth_pct_t_to_t1, 0.5) AS median_growth_t_to_t1
FROM observations
GROUP BY exposure_quartile
ORDER BY exposure_quartile;

## 6. Catálogo y proporción de reseñas

Consulta la dimensión Gold para revisar la distribución de precios, propietarios y valoración. Los valores nulos se conservan como ausencia de información.

In [ ]:
%sql
SELECT is_unreleased,
       COUNT(*) AS games,
       AVG(price) AS mean_price,
       AVG(review_ratio) AS mean_review_ratio,
       COUNT_IF(review_ratio IS NULL) AS games_without_review_ratio
FROM IDENTIFIER(:catalog || '.' || :gold_schema || '.dim_game')
GROUP BY is_unreleased
ORDER BY is_unreleased;

## 7. Exportación de resultados

Vuelve a ejecutar las consultas principales y guarda una muestra de hasta 200 filas por consulta en `notebooks/data_validations/result_outputs/`.

In [ ]:
from datetime import datetime, timezone
import os

def markdown_value(value):
    if value is None:
        return ""
    return str(value).replace("|", "\\|").replace("\n", " ").replace("\r", " ")

def gold_table(name):
    return f"`{catalog}`.`{gold_schema}`.`{name}`"

queries = [
    ("Cobertura de las tablas Gold", f"SELECT 'game_growth' AS table_name, COUNT(*) AS row_count, COUNT(DISTINCT app_id) AS distinct_games, MIN(month_start) AS first_month, MAX(month_start) AS last_month FROM {gold_table('game_growth')} UNION ALL SELECT 'twitch_steam_relationship', COUNT(*), COUNT(DISTINCT app_id), MIN(month_start), MAX(month_start) FROM {gold_table('twitch_steam_relationship')} UNION ALL SELECT 'dim_game', COUNT(*), COUNT(DISTINCT app_id), NULL, NULL FROM {gold_table('dim_game')}"),
    ("Juegos con mayor crecimiento mensual", f"SELECT app_id, name, month_start, avg_players, prev_avg_players, mom_change_pct, months_since_release FROM {gold_table('game_growth')} WHERE mom_change_pct IS NOT NULL ORDER BY mom_change_pct DESC, app_id, month_start LIMIT 50"),
    ("Evolucion mensual agregada", f"SELECT month_start, COUNT(DISTINCT app_id) AS games_with_measurement, AVG(avg_players) AS mean_avg_players, percentile_approx(avg_players, 0.5) AS median_avg_players, AVG(mom_change_pct) AS mean_mom_change_pct FROM {gold_table('game_growth')} GROUP BY month_start ORDER BY month_start"),
    ("Asociacion entre Twitch y crecimiento posterior", f"SELECT COUNT(*) AS observations, COUNT(DISTINCT app_id) AS distinct_games, AVG(hours_watched) AS mean_hours_watched, AVG(growth_pct_t_to_t1) AS mean_steam_growth_t_to_t1, percentile_approx(growth_pct_t_to_t1, 0.5) AS median_steam_growth_t_to_t1, corr(hours_watched, growth_pct_t_to_t1) AS corr_hours_growth FROM {gold_table('twitch_steam_relationship')} WHERE avg_players_t IS NOT NULL AND avg_players_t1 IS NOT NULL AND growth_pct_t_to_t1 IS NOT NULL"),
    ("Asociacion por cuartil de exposicion en Twitch", f"WITH observations AS (SELECT *, ntile(4) OVER (ORDER BY hours_watched, app_id, month_start) AS exposure_quartile FROM {gold_table('twitch_steam_relationship')} WHERE avg_players_t IS NOT NULL AND avg_players_t1 IS NOT NULL AND growth_pct_t_to_t1 IS NOT NULL) SELECT exposure_quartile, COUNT(*) AS observations, MIN(hours_watched) AS min_hours_watched, MAX(hours_watched) AS max_hours_watched, AVG(growth_pct_t_to_t1) AS mean_growth_t_to_t1, percentile_approx(growth_pct_t_to_t1, 0.5) AS median_growth_t_to_t1 FROM observations GROUP BY exposure_quartile ORDER BY exposure_quartile"),
    ("Catalogo y proporcion de resenas", f"SELECT is_unreleased, COUNT(*) AS games, AVG(price) AS mean_price, AVG(review_ratio) AS mean_review_ratio, COUNT_IF(review_ratio IS NULL) AS games_without_review_ratio FROM {gold_table('dim_game')} GROUP BY is_unreleased ORDER BY is_unreleased"),
]
report = [
    "# Resultados del analisis Gold",
    "",
    f"- Catalogo: `{catalog}`",
    f"- Esquema: `{gold_schema}`",
    f"- Generado en UTC: {datetime.now(timezone.utc).isoformat()}",
    "",
    "Los resultados son descriptivos y representan asociaciones observadas; no demuestran causalidad.",
]
for title, query in queries:
    rows = [row.asDict(recursive=True) for row in spark.sql(query).limit(200).collect()]
    report.extend(["", f"## {title}", f"_Filas exportadas: {len(rows)}; maximo aplicado: 200._", ""])
    if not rows:
        report.append("No se devolvieron filas.")
        continue
    columns = list(rows[0].keys())
    report.append("| " + " | ".join(columns) + " |")
    report.append("| " + " | ".join(["---"] * len(columns)) + " |")
    for row in rows:
        report.append("| " + " | ".join(markdown_value(row.get(column)) for column in columns) + " |")

context_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
marker = "/notebooks"
if marker not in context_path:
    raise RuntimeError(f"No se pudo determinar la raiz del repositorio desde la ruta del notebook: {context_path}")
repo_path = context_path.split(marker, 1)[0]
workspace_repo_path = repo_path if repo_path.startswith("/Workspace") else "/Workspace" + repo_path
output_dir = os.path.join(workspace_repo_path, "notebooks", "data_validations", "result_outputs")
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "resultados_gold_analysis.md")
temp_path = output_path + ".tmp"
with open(temp_path, "w", encoding="utf-8", newline="\n") as handle:
    handle.write("\n".join(report) + "\n")
os.replace(temp_path, output_path)
print(f"Markdown generado: {output_path}; consultas exportadas: {len(queries)}")
